# Alignment Pipeline — Fixed

Fixes: runtime GPU, numpy conflict, audioop, taaldetectie, Drive copy, device check, checkpoint upgrade.

## 0. Runtime check

⚠️ Zorg dat je runtime op **GPU** staat: `Runtime → Change runtime type → T4 GPU`

(Notebook had `accelerator: TPU` — dat werkt niet met CUDA)

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ Geen GPU gevonden! Zet runtime op GPU via Runtime → Change runtime type'
print('✅ GPU:', torch.cuda.get_device_name(0))

## 1. Clone repo

In [ ]:
import os
if not os.path.exists('/content/Video_Analyzer'):
    !git clone https://github.com/Yi-Star32/Video_Analyzer.git /content/Video_Analyzer
%cd /content/Video_Analyzer

## 2. Dependencies

Fix volgorde: whisperx eerst (trekt numpy 2.x), daarna niets dat downgradet.
`audioop-lts` werkt niet op Python 3.12 → gefilterd.

In [ ]:
# Installeer requirements zonder audioop-lts (niet beschikbaar op Python 3.12)
!grep -v 'audioop-lts' requirements.txt > /tmp/requirements_fixed.txt
!pip install -q -r /tmp/requirements_fixed.txt

In [ ]:
# Installeer whisperx (trekt numpy>=2.1 mee)
!pip install -q git+https://github.com/m-bain/whisperx.git

In [ ]:
!pip install -q faiss-cpu

In [ ]:
# Verifieer numpy versie — moet >=2.1.0 zijn voor whisperx
import numpy as np
print('numpy:', np.__version__)
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 1), \
    f'❌ numpy {np.__version__} te oud voor whisperx — herstart runtime en run cellen opnieuw'
print('✅ numpy OK')

## 3. Upgrade Lightning checkpoint (eenmalig, elimineert herhaalde warning)

In [ ]:
!python -m lightning.pytorch.utilities.upgrade_checkpoint \
    /usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin 2>/dev/null || true
print('✅ Checkpoint upgrade klaar (of al up-to-date)')

## 4. Imports

In [ ]:
import sys
import warnings
from pathlib import Path
from tqdm import TqdmWarning

warnings.filterwarnings('ignore', category=TqdmWarning)
warnings.filterwarnings('ignore', message='.*gradient_checkpointing.*')

project_path = '/content/Video_Analyzer'
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio
print('✅ Imports OK')

## 5. Google Drive koppelen + audio naar lokale SSD kopiëren

Drive-reads zijn traag (~50 MB/s). Kopieer audio eenmalig naar `/content/audio/` voor 3-5x snelere verwerking.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive/projecten/Video_Analyzer_data')
LOCAL_AUDIO  = Path('/content/audio')
SONG_PATH    = DRIVE_ROOT / 'output/separated/htdemucs/audio/vocals.wav'

assert DRIVE_ROOT.exists(), f'Drive root niet gevonden: {DRIVE_ROOT}'
assert SONG_PATH.exists(),  f'vocals.wav niet gevonden: {SONG_PATH}'
print('✅ Drive OK, song gevonden')

In [ ]:
# Kopieer episodes naar lokale SSD (alleen als nog niet gedaan)
import shutil

DRIVE_EPISODES = DRIVE_ROOT / 'data/episodes_audio'
LOCAL_AUDIO.mkdir(exist_ok=True)

copied = 0
for src in DRIVE_EPISODES.rglob('audio.wav'):
    dst = LOCAL_AUDIO / src.parent.name / 'audio.wav'
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1

files = sorted(LOCAL_AUDIO.rglob('audio.wav'))
print(f'✅ {len(files)} bestanden beschikbaar lokaal ({copied} nieuw gekopieerd)')

## 6. Models initialiseren

Fixes:
- `language='en'` → geen detectie per bestand (~60s bespaard per episode)
- `AudioEmbeddingPipeline(device=device)` → Wav2Vec2 op GPU
- `compute_type='float16'` → sneller op moderne GPU

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')

# FIX: geef device mee aan pipeline zodat Wav2Vec2 ook op GPU draait
pipeline = AudioEmbeddingPipeline(device=device)

# FIX: language='en' voorkomt detectie per bestand
# FIX: compute_type='float16' voor ~2x snelheid op GPU
aligner = PhonemeAligner(
    device=device,
    whisper_model='base',
    language='en',
    compute_type='float16',
)
print('✅ Models geladen')

## 7. Phoneme index bouwen

In [ ]:
if not files:
    raise RuntimeError('Geen audio bestanden gevonden')

pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)
print('✅ Phoneme index klaar')

## 8. Pipeline uitvoeren + exporteren

In [ ]:
OUTPUT_PATH = str(DRIVE_ROOT / 'output/aligned_output_colab1.wav')

final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
print(f'Output lengte: {len(final_audio)} ms')

export_audio(final_audio, OUTPUT_PATH)
print(f'✅ Opgeslagen: {OUTPUT_PATH}')